# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print main metadata fields
meta = dataset.metadata  # Metadata as object
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, record sets and fields are unique via their `@id`. Here we enumerate the available record sets and, for each, their associated fields/columns.

In [ ]:
# List all available record sets in the dataset
record_sets = list(dataset.record_sets)
print(f"Record sets found ({len(record_sets)}):")
for i, rs in enumerate(record_sets):
    print(f"{i+1}. @id: {rs.id} - {rs.name if hasattr(rs, 'name') else ''}")
    # For each record set, print field ids and names
    print("   Fields (columns):")
    for f in rs.fields:
        print(f"     - {f.id}: {f.name} (type: {f.data_type if hasattr(f, 'data_type') else 'unknown'})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In this section, we select the main protein records set and extract it to a pandas DataFrame.

In [ ]:
# Build a dictionary of dataframes for all record sets
dataframes = {}

# In this dataset, the main table is typically labeled with a descriptive name or the first record set. Adjust as necessary after inspection.
if len(record_sets)==0:
    raise Exception("No record sets found! Check the croissant URL/data.")

for rs in record_sets:
    recs = list(dataset.records(record_set=rs.id))  # Records from @id
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Loaded {df.shape[0]} records from {rs.id}")

# Select a record set to demonstrate; we use the first for now
main_record_set_id = record_sets[0].id
print(f"\nFirst record set: {main_record_set_id}")
print(f"Fields (columns): {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field for EDA. You may need to adjust the field `@id` or name after checking column names above.

In [ ]:
# Choose a numeric field (update based on printed columns if necessary)
# Example candidates: 'coverage_pct', 'peptide_count', 'MW', 'pI', 'normalized_abundance', etc.
df = dataframes[main_record_set_id]
possible_numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
print(f"Numeric fields found: {possible_numeric_fields}")

# For demonstration, select the first numeric field
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric_field = '{numeric_field}'")
else:
    raise Exception("No numeric fields for EDA in this record set.")

# Filter records where numeric_field > threshold
threshold = df[numeric_field].mean() if df[numeric_field].dtype != bool else 1  # Use mean or fixed
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (if available)
categorical_fields = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and c != numeric_field]
if categorical_fields:
    group_field = categorical_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll show a histogram of the selected numeric field and, if grouping is available, a bar plot of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Show means by group if grouping was done
if 'grouped_df' in locals():
    plt.figure(figsize=(8,4))
    grouped_df = grouped_df.sort_values(numeric_field, ascending=False)
    grouped_df.head(20).plot(kind='bar', legend=False)
    plt.title(f"Mean {numeric_field} by {group_field} (top 20)")
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains protein mass spectrometry results with several numeric fields for potential analysis.
- Using `mlcroissant`, we loaded both the metadata and records, and demonstrated standard EDA.
- Filtering and normalization can reveal subsets of highest-abundance proteins, and grouping allows aggregation by protein description or accession fields, depending on the dataset.
- Visualizations (e.g., histograms) show data distribution and help in discovering outliers or patterns.

Continue by selecting relevant fields for your biological or analytical question. See the Croissant metadata (`meta`) for rich contextual attributes such as provenance, licensing, and sample-level details.